# Procesamiento de Datos Educativos - ODS.fit

**Objetivo:** Procesar y consolidar datos de educación básica y media superior en México

**Fecha:** 2026-04-14

**Autores:** ODS.fit (Vanessa Bustos, Gustavo Pérez, Elías Pérez)

## Fuentes de datos originales:
- Educación Básica (2021-2025): SEP - Estadística 911
- Educación Media Superior (2019-2025): SEP - Estadística 911

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Configuración de rutas

Definir las rutas de entrada y salida de datos.

In [ ]:
# Rutas del proyecto
PROJECT_ROOT = Path('..')
DATA_RAW = PROJECT_ROOT / 'datos' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'datos' / 'processed'

# Crear carpetas si no existen
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Carpeta de datos raw: {DATA_RAW}")
print(f"Carpeta de datos procesados: {DATA_PROCESSED}")

## 2. Cargar datos desde carpeta raw

Los archivos CSV originales deben estar en `datos/raw/` con la estructura:
- `basica_y_nms/` - Carpeta con archivos de educación básica y media superior

In [ ]:
# Carpeta con archivos consolidados
folder_path = DATA_RAW / 'basica_y_nms'

# Lista para almacenar DataFrames individuales
df_list = []

# Verificar si la carpeta existe
if folder_path.exists():
    # Iterar sobre archivos CSV en la carpeta
    for file_path in folder_path.glob('*.csv'):
        try:
            print(f"Leyendo: {file_path.name}")
            df = pd.read_csv(file_path, encoding='latin1')
            df_list.append(df)
        except Exception as e:
            print(f"Error al leer {file_path.name}: {e}")
    
    print(f"\nTotal de archivos leídos: {len(df_list)}")
else:
    print(f"⚠️ Carpeta no encontrada: {folder_path}")
    print("Por favor, coloca los archivos CSV en datos/raw/basica_y_nms/")

## 3. Concatenar todos los archivos

In [ ]:
if df_list:
    # Concatenar todos los DataFrames
    df_final = pd.concat(df_list, ignore_index=True)
    
    print(f"Dimensiones del dataset concatenado: {df_final.shape}")
    print(f"\nColumnas disponibles:")
    print(df_final.columns.tolist())
else:
    print("No hay datos para concatenar")

## 4. Agrupar por entidad y nivel

Consolidar los datos sumando valores por estado y nivel educativo.

In [ ]:
# Agrupar y sumar por entidad y nivel
df_final = df_final.groupby(['entidad', 'nivel'], as_index=False).sum()

print(f"Dimensiones después de agrupar: {df_final.shape}")
print(f"\nEstados únicos: {df_final['entidad'].nunique()}")
print(f"Niveles únicos: {df_final['nivel'].unique()}")

## 5. Seleccionar columnas relevantes

In [ ]:
# Seleccionar solo las columnas necesarias
columnas_necesarias = [
    'entidad', 'nivel', 'insc_t', 'hom_t', 'muj_t',
    'egre_hom', 'egre_muj', 'egre_tot',
    'docente_h', 'docente_m', 'tot_doc'
]

df_final = df_final[columnas_necesarias]

print("Columnas seleccionadas:")
print(df_final.columns.tolist())

## 6. Calcular porcentajes de abandono escolar

Fórmula: `abandono = 1 - (egresados / inscritos)`

In [ ]:
# Porcentaje de abandono por género y total
df_final['porc_aban_hom'] = np.where(
    df_final['hom_t'] == 0, 
    1.0,  # Si no hay inscritos, considerar 100% abandono (sin datos)
    1 - df_final['egre_hom'] / df_final['hom_t']
)

df_final['porc_aban_muj'] = np.where(
    df_final['muj_t'] == 0,
    1.0,
    1 - df_final['egre_muj'] / df_final['muj_t']
)

df_final['porc_aban_tot'] = np.where(
    df_final['insc_t'] == 0,
    1.0,
    1 - df_final['egre_tot'] / df_final['insc_t']
)

print("Porcentajes de abandono calculados:")
print(df_final[['nivel', 'porc_aban_tot']].groupby('nivel').mean())

## 7. Agregar nombres de entidades

In [ ]:
# Mapeo de códigos de entidad a nombres
ENTIDADES = {
    1: 'aguascalientes',
    2: 'baja california',
    3: 'baja california sur',
    4: 'campeche',
    5: 'coahuila de zaragoza',
    6: 'colima',
    7: 'chiapas',
    8: 'chihuahua',
    9: 'ciudad de méxico',
    10: 'durango',
    11: 'guanajuato',
    12: 'guerrero',
    13: 'hidalgo',
    14: 'jalisco',
    15: 'méxico',
    16: 'michoacán de ocampo',
    17: 'morelos',
    18: 'nayarit',
    19: 'nuevo león',
    20: 'oaxaca',
    21: 'puebla',
    22: 'querétaro',
    23: 'quintana roo',
    24: 'san luis potosí',
    25: 'sinaloa',
    26: 'sonora',
    27: 'tabasco',
    28: 'tamaulipas',
    29: 'tlaxcala',
    30: 'veracruz de ignacio de la llave',
    31: 'yucatán',
    32: 'zacatecas'
}

# Agregar nombres de entidades
df_final['n_entidad'] = df_final['entidad'].map(ENTIDADES)

print("Nombres de entidades agregados:")
print(df_final[['entidad', 'n_entidad']].drop_duplicates().head())

## 8. Calcular porcentajes adicionales

In [ ]:
# Porcentaje de docentes por género
df_final['porc_doc_hom'] = np.where(
    df_final['tot_doc'] == 0,
    0,
    df_final['docente_h'] / df_final['tot_doc']
)

df_final['porc_doc_muj'] = np.where(
    df_final['tot_doc'] == 0,
    0,
    df_final['docente_m'] / df_final['tot_doc']
)

# Porcentaje de inscripciones por género
df_final['porc_ins_hom'] = np.where(
    df_final['insc_t'] == 0,
    0,
    df_final['hom_t'] / df_final['insc_t']
)

df_final['porc_insc_muj'] = np.where(
    df_final['insc_t'] == 0,
    0,
    df_final['muj_t'] / df_final['insc_t']
)

print("Porcentajes adicionales calculados")

## 9. Ordenar columnas en orden final

In [ ]:
# Orden final de columnas
columnas_finales = [
    'entidad', 'n_entidad', 'nivel',
    'insc_t', 'hom_t', 'muj_t',
    'egre_hom', 'egre_muj', 'egre_tot',
    'docente_h', 'docente_m', 'tot_doc',
    'porc_aban_hom', 'porc_aban_muj', 'porc_aban_tot',
    'porc_doc_hom', 'porc_doc_muj',
    'porc_ins_hom', 'porc_insc_muj'
]

df_final = df_final[columnas_finales]

print(f"Dataset final: {df_final.shape}")
print(f"\nPrimeras filas:")
df_final.head()

## 10. Verificar calidad de datos

In [ ]:
print("=== VERIFICACIÓN DE CALIDAD ===")
print(f"\nValores nulos por columna:")
print(df_final.isnull().sum())

print(f"\nRango de entidades: {df_final['entidad'].min()} - {df_final['entidad'].max()}")
print(f"Total de estados: {df_final['entidad'].nunique()}")
print(f"\nNiveles educativos:")
print(df_final['nivel'].value_counts())

print(f"\nEstadísticas descriptivas:")
df_final[['insc_t', 'tot_doc', 'egre_tot']].describe()

## 11. Guardar dataset procesado

In [ ]:
# Guardar el dataset final
output_file = DATA_PROCESSED / 'Datos.csv'
df_final.to_csv(output_file, index=False)

print(f"✓ Dataset guardado en: {output_file}")
print(f"✓ Tamaño del archivo: {output_file.stat().st_size / 1024:.2f} KB")
print(f"✓ Total de registros: {len(df_final)}")

## Resumen del procesamiento

Este notebook:
1. ✓ Lee múltiples archivos CSV de educación básica y media superior
2. ✓ Concatena y agrupa datos por estado y nivel educativo
3. ✓ Calcula porcentajes de abandono escolar por género
4. ✓ Agrega nombres de entidades
5. ✓ Calcula porcentajes de distribución de género en docentes e inscripciones
6. ✓ Genera el archivo final `Datos.csv` con 19 variables

**Output:** `datos/processed/Datos.csv`